# Capital Bikeshare: statistics, probability and sampling
**Live practical · Session 15**

An operations analyst wants to know whether a headline daily-rental average describes a typical day, how much daily demand varies, and what changes when only some days are selected. Work through the two checkpoints. The other cells are ready-to-run demonstrations; you can inspect them without writing plotting or simulation code from scratch.

## Before the first checkpoint: the observed days

Put the approved `day.csv` beside this notebook and open the notebook from that folder. In Colab, upload `day.csv` to the current working folder first; no Drive mount or live dataset download is needed. The data and [source notes](S15_Bike_Sharing_Source_Notes.md) come from Fanaee-T, H. (2013), *Bike Sharing*, UCI Machine Learning Repository, [DOI 10.24432/C5W894](https://doi.org/10.24432/C5W894), under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The CSV is unchanged.

One row represents **one observed calendar day** in the Capital Bikeshare system during 2011–2012. Here the entire file is our finite historical frame (`N` days); a selected subset has `n` days. Describing the frame does not automatically describe future years or other cities. `cnt` is a discrete daily rental count. `season`, `workingday` and `weathersit` are coded categories, even though their cells contain integers. `temp` and `atemp` are normalized measurements, not raw °C; `hum` is normalized humidity, not an unscaled percentage. Keep season codes for calculations because the UCI page and bundled Readme disagree on their labels.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("day.csv")
EXPECTED_SHA256 = "a6bcf826782d3c0fbfdcbeead17cd0884185a0dafe8ff10cd48a874ee7ba18be"
if not DATA_PATH.is_file():
    raise FileNotFoundError("Place the approved day.csv beside this notebook, then rerun this cell.")
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == EXPECTED_SHA256, "Unexpected day.csv bytes"
days = pd.read_csv(DATA_PATH, parse_dates=["dteday"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
days[["dteday", "cnt", "casual", "registered", "season", "workingday", "temp"]].head(3)

,dteday,cnt,casual,registered,season,workingday,temp
0,2011-01-01,985,331,654,1,0,0.344167
1,2011-01-02,801,131,670,1,0,0.363478
2,2011-01-03,1349,120,1229,1,1,0.196364


A **mean** is a sum divided by a count. A **median** is the middle ordered value. A **mode** is the most frequent value or category. A **quartile** marks a position in ordered data; the 90th percentile is a higher position. **Variance** averages squared distances from the mean, so its unit is squared rentals. **Standard deviation (SD)** takes the square root and returns to daily-rental units. For the complete finite frame below, variance and SD divide by `N` (`ddof=0`). We can call that frame variance `σ²` (“sigma squared”) and its SD `σ` (“sigma”). A selected sample's variance/SD for estimating a wider group instead divides by `n−1` (`ddof=1`); its symbols are `s²` and `s`. Pandas defaults to `ddof=1`, so we state the choice explicitly.

A quick classroom trace uses the **illustrative**, non-bike-share values `[5, 10, 15, 20, 25]`. Their mean is 15. Subtract 15 from each value, square those differences and add them: 250. Population variance is 250/5 = 50 squared units and SD is √50 ≈ 7.07 original units. If these five values were a sample used to estimate a wider variance, sample variance is 250/4 = 62.5 and sample SD is √62.5 ≈ 7.91. The division convention changes the question being answered.

In [2]:
# Prepared arithmetic check; these are invented teaching values, not bike-share records.
tiny = np.array([5, 10, 15, 20, 25])
tiny_mean = tiny.mean()
tiny_trace = pd.DataFrame({"value": tiny, "deviation from 15": tiny - tiny_mean,
                           "squared deviation": (tiny - tiny_mean) ** 2})
display(tiny_trace)
print(f"Squared-deviation sum: {tiny_trace['squared deviation'].sum():.0f}; "
      f"population variance/SD: {tiny.var(ddof=0):.1f} / {tiny.std(ddof=0):.2f}; "
      f"sample variance/SD: {tiny.var(ddof=1):.1f} / {tiny.std(ddof=1):.2f}")

,value,deviation from 15,squared deviation
0,5,-10.0,100.0
1,10,-5.0,25.0
2,15,0.0,0.0
3,20,5.0,25.0
4,25,10.0,100.0


Squared-deviation sum: 250; population variance/SD: 50.0 / 7.07; sample variance/SD: 62.5 / 7.91


Two more **illustrative**, non-source comparisons show why an average needs context. Adding one large value to `[10, 15, 20, 25, 30, 20]` raises its mean from 20 to about 45.71 while its median remains 20 in this example; a median is relatively robust, not guaranteed to stay unchanged after every edit. `[48, 49, 50, 51, 52]` and `[30, 40, 50, 60, 70]` both average 50 but have different spread. The mode in the first list is 20. We now inspect the actual daily records.

In [3]:
baseline = pd.Series([10, 15, 20, 25, 30, 20])
with_large_value = pd.concat([baseline, pd.Series([200])], ignore_index=True)
close = pd.Series([48, 49, 50, 51, 52])
wide = pd.Series([30, 40, 50, 60, 70])
print(f"Outlier example: mean {baseline.mean():.2f} → {with_large_value.mean():.2f}; "
      f"median {baseline.median():.0f} → {with_large_value.median():.0f}; "
      f"mode {baseline.mode().tolist()}")
print(f"Equal means: {close.mean():.0f} and {wide.mean():.0f}; "
      f"population SDs: {close.std(ddof=0):.2f} and {wide.std(ddof=0):.2f}")

Outlier example: mean 20.00 → 45.71; median 20 → 20; mode [20]
Equal means: 50 and 50; population SDs: 1.41 and 14.14


## Checkpoint 1 · What do the days reveal? (about 14 minutes)

**C1.1 — Frame and field meanings.** Run the inspection cell. In one sentence each, say what one row represents and what the full observed frame includes. Which listed fields are counts, which are categories, and which weather fields are normalized? Check whether any missing value, duplicate day or count-component mismatch needs attention.

In [ ]:
key_fields = ["dteday", "cnt", "casual", "registered", "season", "workingday", "weathersit", "temp", "hum"]
print("Rows, columns:", days.shape)
print("Date coverage:", days["dteday"].min().date(), "to", days["dteday"].max().date())
print("Key data types:", days[key_fields].dtypes.astype(str).to_dict())
print("Missing cells:", int(days.isna().sum().sum()))
print("Duplicate dates:", int(days["dteday"].duplicated().sum()))
print("Count identity failures:", int((days["cnt"] != days["casual"] + days["registered"]).sum()))
print("Observed category codes:", {name: sorted(days[name].unique().tolist())
                                 for name in ["season", "workingday", "weathersit"]})

**C1.2 — Sample selection.** Run the prepared sampling cell and compare mean and median. `random_50` selects 50 distinct days without replacement using seed 1501. `first_50` takes the first 50 dates. The full-frame mean over `N=731` days can be called `μ` (“mu”); a 50-day sample mean is `x̄` (“x-bar”) over `n=50`. Each `xᵢ` is one day's count and `Σ` means “add them.” Which selection allowed any frame day to enter? Why might the other selection misrepresent the full period? A random sample can still vary by chance; it is not guaranteed to beat every convenience sample.

In [ ]:
SAMPLE_SEED = 1501
sample_rng = np.random.default_rng(SAMPLE_SEED)
chosen_rows = sample_rng.choice(len(days), size=50, replace=False)
random_50 = days.iloc[chosen_rows]
first_50 = days.iloc[:50]
comparison = pd.DataFrame({
    "selection": ["Full observed frame", "Simple random 50", "First 50 dates"],
    "days": [len(days), len(random_50), len(first_50)],
    "mean daily rentals": [group["cnt"].mean() for group in [days, random_50, first_50]],
    "median daily rentals": [group["cnt"].median() for group in [days, random_50, first_50]],
})
display(comparison.round(2))
print("First-50 season codes:", sorted(first_50["season"].unique().tolist()))

**C1.3 — Centre, spread and shape.** Complete the seven short expressions in the next cell. Use `days["cnt"]` for the full frame. For quantiles, use Pandas' **linear interpolation**: the value can fall between observed counts, so it does not promise an exact percentage strictly below it. For the complete frame use `ddof=0`. The prepared chart follows even before you fill the calculation cell. In its boxplot, the line is the median, the triangle is the mean, and the box spans Q1–Q3. Describe one thing a single average conceals.

In [ ]:
frame_counts = days["cnt"]
# Replace each None with a short expression using frame_counts.
frame_mean = None       # TODO: mean
frame_median = None     # TODO: median
q1 = None              # TODO: 0.25 quantile, interpolation="linear"
q3 = None              # TODO: 0.75 quantile, interpolation="linear"
p90 = None             # TODO: 0.90 quantile, interpolation="linear"
frame_variance = None  # TODO: variance, ddof=0
frame_sd = None        # TODO: SD, ddof=0
if any(value is None for value in [frame_mean, frame_median, q1, q3, p90, frame_variance, frame_sd]):
    print("C1.3: complete the seven short expressions above, then rerun this cell.")
else:
    summary = pd.Series({
        "Mean daily rentals": frame_mean, "Median daily rentals": frame_median,
        "Q1 daily rentals": q1, "Q3 daily rentals": q3, "P90 daily rentals": p90,
        "Min daily rentals": frame_counts.min(), "Max daily rentals": frame_counts.max(),
        "Range daily rentals": frame_counts.max() - frame_counts.min(),
        "Population variance (rentals²)": frame_variance,
        "Population SD (daily rentals)": frame_sd, "IQR daily rentals": q3 - q1,
    })
    display(summary.round(2).to_frame("Full 731-day frame"))
    print("Most frequent season code:", days["season"].mode().tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), layout="constrained")
axes[0].hist(days["cnt"], bins=24, color="#226C8A", edgecolor="white")
axes[0].axvline(days["cnt"].mean(), color="#BC4B2C", linewidth=2, label="Frame mean")
axes[0].axvline(days["cnt"].median(), color="#334E68", linestyle="--", linewidth=2,
                label="Frame median")
axes[0].set(title="Observed daily rental counts", xlabel="Daily rentals (cnt)", ylabel="Observed days")
axes[0].legend(frameon=False)
axes[1].boxplot(days["cnt"], widths=0.4, showmeans=True)
axes[1].set(title="Spread of observed days", xticks=[1], xticklabels=["2011–2012 frame"],
            ylabel="Daily rentals (cnt)")
plt.show()

**C1.4 — Sampling design judgement.** In a sentence, compare the simple random 50 with the first 50 dates. How would you ensure all season codes are represented while still randomizing within each group? A systematic method could take every kth day after a random start; a cluster method could choose blocks of days. What periodicity or within-block similarity might matter? Keep this to recognition and design reasoning—no extra sampling code is required.

## Prepared bridge: events and distributions

An **event** is a clear condition. An empirical proportion is a count of matching observed cases divided by the count of cases in its stated reference group. The denominator changes when we condition on another event: `P(A | B)` reads “the proportion of A among B cases.” Complement means “not A.” For an overlap-aware union, count A plus B minus their intersection once. Mutually exclusive events cannot occur together; they are not thereby independent. This little fair-die example is a probability illustration, **not** a claim that daily bike demand follows a die model.

In [ ]:
die = set(range(1, 7))
A = {2, 4, 6}  # even
B = {4, 5, 6}  # greater than 3
print("P(A)=", len(A)/6, "P(not A)=", len(die-A)/6)
print("P(A and B)=", len(A & B)/6, "P(A or B)=", len(A | B)/6)
print("P(A | B)=", len(A & B)/len(B), "(denominator: B outcomes)")
print("A and B are not independent here because P(A | B) differs from P(A).")

A **PMF** assigns probability mass to each discrete outcome; its masses add to 1. A **PDF** is a continuous model's density curve; probability is *area over an interval*, not the curve's height at one exact value. Daily rental counts are discrete integers, even when their binned histogram looks smooth. The normal curve below is only an idealized reference. Its approximate 68–95–99.7 rule applies when a normal model is appropriate; it is not automatically a rule for observed bike-share counts.

In [ ]:
outcomes = np.arange(1, 7)
masses = np.full(6, 1/6)
x = np.linspace(-4, 4, 500)
normal_density = np.exp(-x**2/2) / np.sqrt(2*np.pi)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout="constrained")
axes[0].bar(outcomes, masses, color="#226C8A", width=0.65)
axes[0].set(title="Illustrative fair die: PMF", xlabel="Discrete face", ylabel="Probability mass",
            xticks=outcomes, ylim=(0, 0.22))
axes[1].plot(x, normal_density, color="#334E68", linewidth=2)
axes[1].fill_between(x, normal_density, where=(x >= -1) & (x <= 1), color="#7DB9A7",
                     alpha=0.7, label="Area from −1 to +1 ≈ 0.68")
axes[1].set(title="Illustrative normal model: PDF", xlabel="Standardized model value",
            ylabel="Probability density")
axes[1].legend(frameon=False)
plt.show()
print("Die PMF mass total:", round(masses.sum(), 2))

## Prepared bridge: repeated sample means

The next cell resamples the **finite observed frame** with replacement, using seed 1516, for 2,000 repetitions at each size `n=5` and `n=50`. Each repetition gives one *sample mean*; these means form a sampling distribution. Replacement makes the simulated draws independent **within this empirical exercise**, but the original dated records may have temporal dependence and seasonality. This demonstration does not create new future observations or prove that raw rentals are normal. Under suitable independent-draw and finite-variance conditions, larger-sample mean distributions can become more nearly normal and less variable; there is no universal magic sample size.

In [ ]:
SIM_SEED = 1516
REPEATS = 2000
values = days["cnt"].to_numpy()
sim_rng = np.random.default_rng(SIM_SEED)
means_5 = sim_rng.choice(values, size=(REPEATS, 5), replace=True).mean(axis=1)
means_50 = sim_rng.choice(values, size=(REPEATS, 50), replace=True).mean(axis=1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharex=True, layout="constrained")
for ax, data, title, ylabel in zip(
    axes, [values, means_5, means_50],
    ["Raw observed days", "Simulated means: n=5", "Simulated means: n=50"],
    ["Observed days", "Replicates", "Replicates"]):
    ax.hist(data, bins=np.linspace(0, 9000, 31), color="#226C8A", edgecolor="white")
    ax.axvline(values.mean(), color="#BC4B2C", linewidth=2, label="Frame mean")
    ax.set(title=title, xlabel="Daily rentals or mean daily rentals", ylabel=ylabel, xlim=(0, 9000))
axes[2].legend(frameon=False, loc="upper right")
plt.show()
print("SD of raw days:", round(values.std(ddof=0), 1), "daily rentals")
print("SD of simulated sample means (n=5, n=50):",
      round(means_5.std(ddof=0), 1), round(means_50.std(ddof=0), 1), "daily rentals")

## Checkpoint 2 · Can we responsibly explain demand? (about 19 minutes)

**C2.1 — Shape and reporting.** Revisit the actual count histogram and your C1.3 results. Is there a strong longer tail in one direction, or a roughly balanced broad shape? Does a mean alone convey the variation? Would you add SD, median/IQR or both? Identify an extreme observed day without calling it a data error. Skew names the **longer tail**, not the peak.

**C2.2 — Fixed events and conditional denominators.** `H` means `cnt >= 5000`; `W` means `workingday == 1`. The table's `True`/`False` row and column labels correspond to those exact conditions. The proportion `P(H)` uses all observed days as denominator. `P(H | W)` uses only working days. `P(H | not W)` uses only nonworking days. These are empirical historical proportions, not causal effects or guaranteed future chances.

In [ ]:
H = days["cnt"] >= 5000
W = days["workingday"] == 1
event_table = pd.crosstab(W, H, rownames=["W: workingday == 1"],
                          colnames=["H: cnt >= 5000"]).reindex(
                              index=[False, True], columns=[False, True], fill_value=0)
display(event_table)

In [ ]:
# Read the table or masks above. Replace four None values with integer counts.
h_count = None        # TODO: number of H days
w_count = None        # TODO: number of W days
h_and_w = None        # TODO: number of H and W days
h_and_not_w = None    # TODO: number of H and not-W days
if any(value is None for value in [h_count, w_count, h_and_w, h_and_not_w]):
    print("C2.2: fill the four counts above, then rerun to see numerators and denominators.")
else:
    not_w_count = len(days) - w_count
    print(f"P(H) = {h_count}/{len(days)} = {h_count/len(days):.4f}")
    print(f"P(H | W) = {h_and_w}/{w_count} = {h_and_w/w_count:.4f}")
    print(f"P(H | not W) = {h_and_not_w}/{not_w_count} = {h_and_not_w/not_w_count:.4f}")
    print(f"P(H and W) = {h_and_w}/{len(days)} = {h_and_w/len(days):.4f}")
    union_count = h_count + w_count - h_and_w  # count the overlap once
    print(f"P(H or W) = {union_count}/{len(days)} = {union_count/len(days):.4f}")
    print(f"P(not H) = {len(days)-h_count}/{len(days)} = {(len(days)-h_count)/len(days):.4f}")

Explain why the denominator changes from all days to only working days. The two primary rates may be close—why does closeness alone not prove independence? For `H or W`, which overlap must be subtracted once? Do not turn a historical association into a causal statement.

**C2.3 — Standardized position.** A z-score reads as a *signed number of SD units* from a stated reference mean: `(observed count − frame mean) / frame SD`. Here the reference is all 731 historical days and `ddof=0`. We use the actual day **2012-09-15**, selected deterministically from the CSV. A z-score does not prove a normal distribution or supply an automatic normal percentile.

In [ ]:
example_day = days.loc[days["dteday"].eq(pd.Timestamp("2012-09-15"))].iloc[0]
reference_mean = days["cnt"].mean()
reference_sd = days["cnt"].std(ddof=0)
z_score = None  # TODO: subtract reference_mean from this day's cnt, then divide by reference_sd
if z_score is None:
    print(f"C2.3: selected day has {example_day['cnt']} rentals. Complete z_score, then rerun.")
else:
    print(f"{example_day['dteday'].date()}: {example_day['cnt']} rentals; "
          f"z = ({example_day['cnt']} − {reference_mean:.2f})/{reference_sd:.2f} = {z_score:.3f}")

State whether this day lies above or below the frame mean and how many frame SDs away. Does that result alone establish a normal percentile? Use the real count chart as a reason to check shape rather than assume normality.

**C2.4 — Read the repeated-sampling display.** Which panel shows actual daily records? Which two show sample means? Compare the widths for `n=5` and `n=50`. Why do the means still vary? Does the display make the original daily counts normal or prove future days are independent? The simulation uses replacement and a fixed seed so you can rerun it reliably.

**C2.5 — Your analyst's note.** Write 4–6 sentences. Include: (1) the 2011–2012 historical scope and a typical daily count; (2) one numerical spread measure and a careful shape description; (3) `P(H | W)` as a count over the **working-day** denominator; (4) a selection or coverage caveat; and (5) one question about a wider group that later inference methods could investigate. Use your executed results, not guessed numbers.

**Your note:**

> [Write here.]

## Take forward

Before reporting a statistic, name its observation unit, reference group and selection method. This practical describes the historical daily frame and demonstrates what a selected sample can miss. The next session can ask how to quantify uncertainty about a carefully defined wider target; no such inference is performed here.